# IMDB Film Yorumları Duygu Analizi - BERT ile Gelişmiş Sürüm

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 1. GENEL AYARLAR VE PARAMETRE TERCİHLERİ

In [ ]:
class Config:
    """Model ve eğitim parametrelerini tutar"""
    # Model parametreleri
    MODEL_NAME = 'bert-base-uncased'
    MAX_SEQUENCE_LENGTH = 128
    NUM_CLASSES = 2

    # Eğitim parametreleri
    EPOCHS = 3
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5

    # Veri parametreleri
    TRAIN_SIZE = 0.8  # %80 eğitim, %20 test
    RANDOM_STATE = 42

config = Config()

# 2. DONANIM OPTIMIZASYONU VE STRATEJI BELIRLEME

In [ ]:
def setup_hardware_strategy():
    """En uygun donanım stratejisini belirler"""
    print("🔧 Donanım durumu kontrol ediliyor...")

    # TPU kontrolü
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.TPUStrategy(tpu)
        print("✅ TPU aktif! Hızlı eğitim modu açık.")
        return strategy
    except ValueError:
        print("⚠️  TPU bulunamadı.")

    # GPU kontrolü
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"✅ {len(gpus)} GPU bulundu!")
        strategy = tf.distribute.MirroredStrategy()
        return strategy
    else:
        print("💻 CPU modunda çalışılıyor.")
        return tf.distribute.get_strategy()

# 3. VERİ YÜKLEME VE İNCELEME

In [ ]:
def load_and_explore_data():
    """IMDB veri setini yükler ve inceler"""
    print("\n📊 Veri seti yükleniyor...")

    # Veri setini indir
    url = 'https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/refs/heads/master/IMDB-Dataset.csv'
    df = pd.read_csv(url, dtype=str)

    print(f"✅ Veri seti yüklendi! Toplam {len(df):,} yorum")

    # Veri setini incele
    print("\n📈 Veri Seti İncelemesi:")
    print(f"   • Sütunlar: {df.columns.tolist()}")
    print(f"   • Boyut: {df.shape}")
    print(f"   • Eksik değer: {df.isnull().sum().sum()}")

    # Duygu dağılımı
    sentiment_counts = df['sentiment'].value_counts()
    print(f"\n📊 Duygu Dağılımı:")
    for sentiment, count in sentiment_counts.items():
        print(f"   • {sentiment}: {count:,} ({count/len(df)*100:.1f}%)")

    return df

# 4. VERİ ÖN İŞLEME VE HAZIRLIK

In [ ]:
def prepare_data(df):
    """Veriyi eğitim ve test için hazırlar"""
    print("\n🔄 Veri ön işleme başlıyor...")

    # Sütun isimlerini düzenle
    df.columns = df.columns.str.lower().str.strip()

    # Metin temizliği
    df['review'] = df['review'].str.strip()
    df['review'] = df['review'].fillna('')

    # Etiketleri sayısal değerlere çevir
    label_mapping = {'positive': 1, 'negative': 0}
    df['label'] = df['sentiment'].map(label_mapping)

    # Veriyi karıştır
    df = df.sample(frac=1, random_state=config.RANDOM_STATE).reset_index(drop=True)

    # Eğitim ve test verilerini ayır
    split_index = int(len(df) * config.TRAIN_SIZE)

    train_df = df[:split_index].copy()
    test_df = df[split_index:].copy()

    print(f"✅ Veri hazırlandı!")
    print(f"   • Eğitim seti: {len(train_df):,} yorum")
    print(f"   • Test seti: {len(test_df):,} yorum")

    return train_df, test_df

# 5. METİN TOKENİZASYONU

In [ ]:
def setup_tokenizer():
    """BERT tokenizer'ını hazırlar"""
    print("\n🔤 Tokenizer hazırlanıyor...")
    tokenizer = BertTokenizer.from_pretrained(config.MODEL_NAME)
    print("✅ Tokenizer hazır!")
    return tokenizer

def tokenize_data(texts, labels, tokenizer, max_length):
    """Metinleri BERT için tokenize eder"""
    print(f"🔄 {len(texts):,} metin tokenize ediliyor...")

    # Her metni tokenize et
    encoded = tokenizer(
        texts.tolist(),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_attention_mask=True,
        return_tensors='np'
    )

    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'labels': np.array(labels)
    }

# 6. MODEL OLUŞTURMA VE DERLEME

In [ ]:
def create_bert_model(strategy):
    """BERT modelini oluşturur ve derler"""
    print("\n🧠 BERT modeli oluşturuluyor...")

    with strategy.scope():
        # Önceden eğitilmiş BERT modelini yükle
        model = TFBertForSequenceClassification.from_pretrained(
            config.MODEL_NAME,
            num_labels=config.NUM_CLASSES
        )

        # Optimizasyon parametreleri
        optimizer = tf.keras.optimizers.Adam(learning_rate=config.LEARNING_RATE)
        loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        metrics = [tf.keras.metrics.SparseCategoricalAccuracy('accuracy')]

        # Modeli derle
        model.compile(
            optimizer=optimizer,
            loss=loss,
            metrics=metrics
        )

    print("✅ Model hazır!")
    print(f"   • Parametre sayısı: {model.count_params():,}")

    return model

# 7. MODEL EĞİTİMİ

In [ ]:
def train_model(model, train_data, test_data):
    """Modeli eğitir"""
    print(f"\n🚀 Model eğitimi başlıyor... ({config.EPOCHS} epoch)")

    # Eğitim geçmişini kaydet
    start_time = datetime.now()

    history = model.fit(
        [train_data['input_ids'], train_data['attention_mask']],
        train_data['labels'],
        epochs=config.EPOCHS,
        batch_size=config.BATCH_SIZE,
        validation_data=(
            [test_data['input_ids'], test_data['attention_mask']],
            test_data['labels']
        ),
        verbose=1
    )

    end_time = datetime.now()
    training_time = end_time - start_time

    print(f"✅ Eğitim tamamlandı! Süre: {training_time}")

    return history

# 8. MODEL DEĞERLENDİRME

In [ ]:
def evaluate_model(model, test_data):
    """Modelin performansını değerlendirir"""
    print("\n📊 Model değerlendiriliyor...")

    # Test verisiyle değerlendirme
    test_loss, test_accuracy = model.evaluate(
        [test_data['input_ids'], test_data['attention_mask']],
        test_data['labels'],
        verbose=0
    )

    print(f"📈 Test Sonuçları:")
    print(f"   • Doğruluk: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    print(f"   • Kayıp: {test_loss:.4f}")

    # Tahminleri al
    predictions = model.predict([test_data['input_ids'], test_data['attention_mask']])[0]
    predicted_labels = np.argmax(predictions, axis=1)

    # Detaylı rapor
    print("\n📋 Detaylı Sınıflandırma Raporu:")
    print(classification_report(
        test_data['labels'],
        predicted_labels,
        target_names=['Negatif', 'Pozitif']
    ))

    return predicted_labels

# 9. TAHMİN FONKSİYONU

In [ ]:
def create_prediction_function(model, tokenizer):
    """Yeni metinler için tahmin fonksiyonu oluşturur"""

    def predict_sentiment(text, return_confidence=False):
        """
        Verilen metin için duygu tahmini yapar

        Args:
            text (str): Analiz edilecek metin
            return_confidence (bool): Güven skorunu da döndür

        Returns:
            str veya tuple: Tahmin edilen duygu (ve güven skoru)
        """
        # Metni tokenize et
        encoded = tokenizer(
            [text],
            truncation=True,
            padding='max_length',
            max_length=config.MAX_SEQUENCE_LENGTH,
            return_tensors='np'
        )

        # Tahmin yap
        prediction = model.predict([encoded['input_ids'], encoded['attention_mask']])[0]
        predicted_class = np.argmax(prediction[0])
        confidence = np.max(tf.nn.softmax(prediction[0]).numpy())

        sentiment = "Pozitif" if predicted_class == 1 else "Negatif"

        if return_confidence:
            return sentiment, confidence
        return sentiment

    return predict_sentiment

# 10. ANA PROGRAM

In [ ]:
def main():
    """Ana program akışı"""
    print("🎬 IMDB Film Yorumları Duygu Analizi")
    print("=" * 50)

    # 1. Donanım stratejisi
    strategy = setup_hardware_strategy()

    # 2. Veriyi yükle ve incele
    df = load_and_explore_data()

    # 3. Veriyi hazırla
    train_df, test_df = prepare_data(df)

    # 4. Tokenizer'ı hazırla
    tokenizer = setup_tokenizer()

    # 5. Verileri tokenize et
    train_data = tokenize_data(
        train_df['review'],
        train_df['label'],
        tokenizer,
        config.MAX_SEQUENCE_LENGTH
    )

    test_data = tokenize_data(
        test_df['review'],
        test_df['label'],
        tokenizer,
        config.MAX_SEQUENCE_LENGTH
    )

    # 6. Modeli oluştur
    model = create_bert_model(strategy)

    # 7. Modeli eğit
    history = train_model(model, train_data, test_data)

    # 8. Modeli değerlendir
    predictions = evaluate_model(model, test_data)

    # 9. Tahmin fonksiyonunu oluştur
    predict_sentiment = create_prediction_function(model, tokenizer)

    # 10. Örnek tahminler
    print("\n🔮 Örnek Tahminler:")
    sample_reviews = [
        "This movie was absolutely fantastic! I loved every minute of it.",           # Pozitif
        "What a terrible film, completely waste of time and money.",                  # Negatif
        "The acting was brilliant and the storyline kept me engaged throughout.",     # Pozitif
        "Boring and predictable plot, I fell asleep halfway through the movie.",      # Negatif
        "Outstanding performance by the lead actor, definitely worth watching!"       # Pozitif
    ]

    for review in sample_reviews:
        sentiment, confidence = predict_sentiment(review, return_confidence=True)
        print(f"   📝 '{review[:50]}...'")
        print(f"   💭 Tahmin: {sentiment} (Güven: {confidence:.2f})")
        print()

    print("✅ Program tamamlandı!")

    return model, tokenizer, predict_sentiment

# PROGRAMI ÇALIŞTIR

In [ ]:
if __name__ == "__main__":
    model, tokenizer, predict_sentiment = main()